In [67]:
import pandas as pd
import numpy as np
import re
from pathlib import Path 


In [68]:
# Labels for answer choices the data leaves without one: a bare number ('02') or no value label at all (NaN).
# A scale is identified by its labelled choices, in code order; its entry gives the label of every unlabelled code.
# The same scale gets the same labels in any dataset. Labelled choices are never changed.
LABEL_BANK = {
    ("None", "1 child", "2 children"): {
        3: "3 children",
        4: "4 children",
        5: "5 children",
        6: "6 children",
        7: "7 children",
        8: "8 children",
        9: "9 children",
        10: "10 children",
    },
    ("Yes, they definitely should", "No, they definitely should not"): {
        2: "Yes, they should",
        3: "Yes, they probably should",
        4: "Undecided",
        5: "No, they probably should not",
        6: "No, they should not",
    },
    ("You cannot rely on either of them", "You can definitely rely more on close relatives",
     "You can rely equally on both", "You can definitely rely more on close friends"): {
        2: "You can rely more on close relatives",
        3: "You can probably rely more on close relatives",
        5: "You can probably rely more on close friends",
        6: "You can rely more on close friends",
    },
    ("Close relatives should definitely be the most important", "Both should be equally important",
     "Close friends should definitely be the most important"): {
        2: "Close relatives should be the most important",
        3: "Close relatives should probably be the most important",
        5: "Close friends should probably be the most important",
        6: "Close friends should be the most important",
    },
}

BRACKETS = {"[COUNTRY's]": "{country}'s", "[COUNTRY]": "{country}", "[CNTRY]": "{country}",
            "[EU/ USMCA/ etc.]": "{association}"}                       # the data's placeholders -> the bank's

def fill_placeholders(text, fields, where=""):
    """Text with this country's names in place of placeholders: the bank's {country}, {country}'s, {nationality},
    {parliament}, {association}, and the data's own bracket forms (BRACKETS). A 'the' that starts a sentence is
    capitalised. Prints every placeholder filled (where: e.g. the question id), and any it cannot fill."""
    name = fields["country"]
    values = {"{country}'s": name + ("'" if name.endswith("s") else "'s"), **{f"{{{k}}}": v for k, v in fields.items()}}
    for bracket, brace in BRACKETS.items():
        text = text.replace(bracket, brace)
    for placeholder, value in values.items():
        if placeholder in text:
            print(f"{where:<5} {placeholder:<15} ->  {value!r}")
            text = text.replace(placeholder, value)
    text = re.sub(r"(^|[.?!…]\s+)the ", lambda m: m.group(1) + "The ", text)
    for left in re.findall(r"\{[^}]*\}|\[[^\]]*\]", text):
        print(f"{where:<5} {left:<15} ->  UNFILLED (no value for it)")
    return text

def unlabelled(text):
    return not isinstance(text, str) or not re.search(r"[A-Za-z]", text)   # NaN or no letters ('02')

def scale_key(texts):
    return tuple(t for t in texts if not unlabelled(t))                      # labelled choices in code order

def fill_labels(codes, texts):
    fixes = LABEL_BANK.get(scale_key(texts), {})
    return [fixes.get(c, t) if unlabelled(t) else t for c, t in zip(codes, texts)]


COUNTRY_NOTE = re.compile(r"^([A-Z]{2}(?:\s*,\s*[A-Z]{2})*)\s*:\s*")   # 'NL, US: Employers' -> offered only in NL and US

def offered_options(labels, country, verbose=False):
    """{code: text} of every valid (non-negative) option offered in `country`, in code order. An option marked for
    other countries only is dropped; an option marked for `country` keeps its text without the marker.
    verbose=True prints every option it drops or changes."""
    out = {}
    for k, t in sorted(labels.items()):
        if k < 0:
            continue
        text = re.sub(r"^-?\d+\.\s*", "", t)
        m = COUNTRY_NOTE.match(text)
        if m:
            if country not in re.split(r"\s*,\s*", m[1]):
                if verbose:
                    print(f"  {int(k):>4}  {text!r}  ->  dropped: not offered in {country}")
                continue
            if verbose:
                print(f"  {int(k):>4}  {text!r}  ->  {text[m.end():]!r}")
            text = text[m.end():]
        out[int(k)] = text
    return out

def persona_text(label, country):
    """Persona value as this country's respondents saw it: '1. Male' -> 'Male'; 'Once a year; US: Once or twice a year'
    -> 'Once or twice a year' in the US, 'Once a year' elsewhere."""
    text = re.sub(r"^-?\d+\.\s*", "", str(label)).strip()
    kept, ours = [], None
    for part in text.split(";"):
        part = part.strip()
        m = COUNTRY_NOTE.match(part)
        if not m:
            kept.append(part)                          # general wording
        elif country in re.split(r"\s*,\s*", m[1]):
            ours = part[m.end():]                      # this country's own wording
    return ours if ours is not None else "; ".join(kept)

In [69]:
data_dir = Path.cwd()/'data/issp2023'
data_fn = data_dir/'raw/ZA10010_v1-0-0.dta'
bank_fn = data_dir/'question_bank.csv'
verified_fn = data_dir/'questions_verified.csv'

country = 'US'
max_options = 15
countries = pd.read_csv(Path.cwd()/'data/issp_countries.csv', keep_default_na=False).set_index("c_alphan")
fields = {**countries.loc[country, ["country", "nationality", "parliament"]].to_dict(),   # this country's names
          "association": "the United States-Mexico-Canada Agreement (USMCA)"}             # 2023 Q38-Q43: data/issp2023/build_dataset.py

bank = pd.read_csv(bank_fn)

wide = pd.read_stata(data_fn)                                   # as stored; answers shown with GESIS's labels
raw = pd.read_stata(data_fn, convert_categoricals=False)        # NEW: same file, answers as integer codes
labels = pd.io.stata.StataReader(data_fn).variable_labels()     # column -> GESIS label
with pd.io.stata.StataReader(data_fn) as reader:
    label_sets = reader.value_labels()                                   # label-set name -> {code: "N. text"}
    value_labels = {v: label_sets.get(s, {}) for v, s in zip(reader._varlist, reader._lbllist)}   # column -> {code: "N. text"}

wide = wide[wide["c_alphan"] == country]
raw = raw.loc[wide.index]                                       # NEW: same rows as wide
wide["CASEID"] = wide["CASEID"].astype("int64")             # stored as a float; shown as 2.022008e+15 otherwise

persona = ["c_alphan", "SEX", "AGE", "EDULEVEL", "RELIGGRP", "ATTEND", "PARTY_LR", "VOTE_LE", "URBRURAL",
           "MARITAL", "MAINSTAT", "UNION", f"{country}_PRTY", f"{country}_INC", f"{country}_REG", f"{country}_ETHN1"]
questions = [c for c in wide.columns if re.fullmatch(r"v\d+", c)]   # v1, v2, ...: the substantive questions

def all_options(v):
    """{code: text} of every option of question v in this country, in code order: the labelled options offered here
    (offered_options), plus every non-negative code without a label between the labelled ends or given by a respondent
    here (text NaN, filled by LABEL_BANK in the next cell). Codes labelled only for other countries stay out."""
    opts = offered_options(value_labels[v], country)
    given = set(raw.loc[raw[v] >= 0, v].astype(int))                   # codes respondents in this country gave
    labelled = {int(k) for k in value_labels[v] if k >= 0}              # codes with a value label (any country)
    span = set(range(min(opts), max(opts) + 1)) if opts else set()      # every code between the labelled ends
    for c in (span - labelled) | (given - set(opts)):
        opts[c] = float("nan")
    return dict(sorted(opts.items()))

options = {v: all_options(v) for v in questions}                        # v -> {code: text}, every option in this country

wide[questions] = wide[questions].where(raw[questions] >= 0)                          # NEW: negative answers -> NaN (that cell only)

questions = [v for v in questions                                              # the one place questions are kept or dropped:
             if v in set(bank.loc[bank["type"] != "exclude", "v"])            #   in the bank and not marked exclude
             and len(options[v]) <= max_options]                              #   at most max_options offered options

bank = bank[bank["v"].isin(questions)]

# this country's names in place of placeholders, in question texts and option texts
bank["question"] = [fill_placeholders(t, fields, v) for v, t in zip(bank["v"], bank["question"])]
options = {v: {c: fill_placeholders(t, fields, v) if isinstance(t, str) else t for c, t in o.items()} for v, o in options.items()}

wide = wide[["CASEID"] + persona + questions]

wide[questions] = wide[questions].apply(lambda s: s.cat.set_categories(
    [value_labels[s.name].get(k, k) for k in options[s.name]]))         # categories = every option in this country, in code order (an unlabelled code is its own category)

# persona: number prefix removed, this country's wording taken from country notes, negative codes -> NaN
for c in persona[1:]:                                             # c_alphan stay as it is
    vl = value_labels.get(c, {})
    values = {}
    for k in sorted(raw[c].dropna().unique()):
        before = re.sub(r"^-?\d+\.\s*", "", str(vl.get(k, k))).strip()
        if k < 0:
            values[k] = float("nan")
            print(f"{c:<10} {int(k):>6}  {before!r}  ->  NaN  ({int((raw[c] == k).sum())} respondents)")
        else:
            values[k] = persona_text(vl.get(k, k), country)
            if values[k] != before:
                print(f"{c:<10} {int(k):>6}  {before!r}  ->  {values[k]!r}")
    wide[c] = raw[c].map(values)

long = wide.melt(id_vars=["CASEID"]+persona, value_vars=questions, var_name="v", value_name="answer")
long.insert(long.columns.get_loc("v") + 1, "label", long["v"].map(labels))
long["answer_code"] = raw[questions].melt()["value"].astype("Int64").to_numpy()   # CHANGED: code from the raw integers, not the label string
long = long.dropna(subset=["answer"])  # CHANGED: drop the NaN cells (negative or missing answers)

long.head(2)

v1    {country}       ->  'the United States'
v1    {nationality}   ->  'American'
v2    {nationality}   ->  'American'
v3    {country}       ->  'the United States'
v3    {nationality}   ->  'American'
v4    {nationality}   ->  'American'
v5    {nationality}   ->  'American'
v6    {nationality}   ->  'American'
v7    {country}       ->  'the United States'
v8    {country}       ->  'the United States'
v9    {nationality}   ->  'American'
v10   {country}       ->  'the United States'
v13   {nationality}   ->  'American'
v14   {country}       ->  'the United States'
v15   {country}       ->  'the United States'
v16   {country}'s     ->  "the United States'"
v16   {country}       ->  'the United States'
v17   {country}       ->  'the United States'
v18   {country}       ->  'the United States'
v19   {country}       ->  'the United States'
v20   {country}       ->  'the United States'
v21   {country}       ->  'the United States'
v22   {country}       ->  'the United States'
v23   {countr

,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,MAINSTAT,UNION,US_PRTY,US_INC,US_REG,US_ETHN1,v,label,answer,answer_code
0,2023008400000002,US,Male,64,"Upper level tertiary, MA",Protestant,Never,Center / liberal,Yes,The suburbs or outskirts of a big city,...,Retired,No,Democrat,60 000 - 74 999 USD,New England,Czechoslovakia,v1,Q1a How important for being truly [NAT]: To ha...,4. Not important at all,4
1,2023008400000003,US,Female,69,Short-cycle tertiary,No religion,Never,Center / liberal,Yes,The suburbs or outskirts of a big city,...,Retired,No,Democrat,110 000 - 129 999 USD,New England,England and Wales,v1,Q1a How important for being truly [NAT]: To ha...,4. Not important at all,4


In [70]:
# bank: every option offered in this country, in code order; label fixes shown before -> after
info = bank.set_index("v")
rows = []
for v in questions:
    before = {int(k): re.sub(r"^-?\d+\.\s*", "", t) for k, t in sorted(value_labels[v].items()) if k >= 0}   # as the data has it
    before = dict(sorted((before | {c: float("nan") for c in options[v] if c not in before}).items()))      # + codes without a label
    opts = options[v]                                                                                       # country markers handled
    codes = list(opts)
    after = dict(zip(codes, fill_labels(codes, list(opts.values()))))                                       # unlabelled choices filled
    if any(unlabelled(t) for t in after.values()):                     # no LABEL_BANK entry: the data's numbered form for the whole scale
        after = {c: f"{c}. {t}" if not unlabelled(t) else f"{c}." for c, t in after.items()}                                    # unlabelled choices filled
    rows.append({"v": v,
                 "answer_code": codes,
                 "answer": [value_labels[v].get(k, str(k)) for k in codes],
                 "answer_text": list(after.values())})
    if before == after:
        continue
    print("=" * 100)
    print(f"{v} | {info.loc[v, 'type']} | {info.loc[v, 'data_label']}")
    if any(unlabelled(t) for t in after.values()):
        print("NO ENTRY IN LABEL_BANK FOR THIS SCALE")
    print("-" * 100)
    for c, b in before.items():
        print(f"{c:>4}  {str(b):<45} ->  {after.get(c, f'(dropped: not offered in {country})')}")
print("=" * 100)

bank = bank.merge(pd.DataFrame(rows), on="v")

# long: each person's answer text taken from bank, so the two always agree
text_of = {(r["v"], c): t for _, r in bank.iterrows() for c, t in zip(r["answer_code"], r["answer_text"])}
long["answer_text"] = [text_of[(v, c)] for v, c in zip(long["v"], long["answer_code"])]

# long: the question each answer belongs to, with its type, direction and every option offered
long = long.merge(
    bank[["v", "question", "type", "reversed", "answer_code", "answer_text"]]
        .rename(columns={"answer_code": "option_codes", "answer_text": "option_texts"}),
    on="v", how="left")
long.head(2)

v32 | choice | Q9 Opinion on right behaviour of immigrants in terms of retaining their culture 
----------------------------------------------------------------------------------------------------
   1  Immigrants should retain their culture of origin and not adopt [COUNTRY's] culture ->  Immigrants should retain their culture of origin and not adopt the United States' culture
   2  Immigrants should retain their culture of origin and also adopt [COUNTRY's] culture ->  Immigrants should retain their culture of origin and also adopt the United States' culture
   3  Immigrants should give up their culture of origin and adopt [COUNTRY's] culture ->  Immigrants should give up their culture of origin and adopt the United States' culture
v33 | score | Q10 How important for being good citizen: Never to try to evade taxes
NO ENTRY IN LABEL_BANK FOR THIS SCALE
----------------------------------------------------------------------------------------------------
   1  Not at all important         

,CASEID,c_alphan,SEX,AGE,EDULEVEL,RELIGGRP,ATTEND,PARTY_LR,VOTE_LE,URBRURAL,...,v,label,answer,answer_code,answer_text,question,type,reversed,option_codes,option_texts
0,2023008400000002,US,Male,64,"Upper level tertiary, MA",Protestant,Never,Center / liberal,Yes,The suburbs or outskirts of a big city,...,v1,Q1a How important for being truly [NAT]: To ha...,4. Not important at all,4,Not important at all,Some people say that the following things are ...,score,True,"[1, 2, 3, 4]","[Very important, Fairly important, Not very im..."
1,2023008400000003,US,Female,69,Short-cycle tertiary,No religion,Never,Center / liberal,Yes,The suburbs or outskirts of a big city,...,v1,Q1a How important for being truly [NAT]: To ha...,4. Not important at all,4,Not important at all,Some people say that the following things are ...,score,True,"[1, 2, 3, 4]","[Very important, Fairly important, Not very im..."


In [71]:
# for q in long['question'].unique():
#     print(q)
#     print('+'*50)

In [72]:
qid = 'v38'
# qid = 'v46'
long[long['v']==qid].answer.value_counts()

answer
5                 308
7                 228
6                 215
4                 157
8                 138
3                 121
0. Very poorly    109
10. Very well      84
2                  64
9                  37
1                  29
Name: count, dtype: int64

In [73]:
import random

n = 2
long
persona
persona_cols = random.choices(persona, k=n)
long[['v']+persona_cols+['question', 'answer_code', 'option_texts']]


,v,VOTE_LE,RELIGGRP,question,answer_code,option_texts
0,v1,Yes,Protestant,Some people say that the following things are ...,4,"[Very important, Fairly important, Not very im..."
1,v1,Yes,No religion,Some people say that the following things are ...,4,"[Very important, Fairly important, Not very im..."
2,v1,Yes,Catholic,Some people say that the following things are ...,2,"[Very important, Fairly important, Not very im..."
3,v1,No,No religion,Some people say that the following things are ...,3,"[Very important, Fairly important, Not very im..."
4,v1,No,Other Religions,Some people say that the following things are ...,2,"[Very important, Fairly important, Not very im..."
...,...,...,...,...,...,...
89386,v62,Yes,No religion,"Generally speaking, would you say that the Uni...",4,[Have never heard of the United States-Mexico-...
89387,v62,No,No religion,"Generally speaking, would you say that the Uni...",3,[Have never heard of the United States-Mexico-...
89388,v62,Yes,Protestant,"Generally speaking, would you say that the Uni...",3,[Have never heard of the United States-Mexico-...
89389,v62,Yes,Catholic,"Generally speaking, would you say that the Uni...",3,[Have never heard of the United States-Mexico-...


In [74]:
missing = bank[bank["answer_text"].apply(
    lambda texts: any(not isinstance(t, str) or not re.search(r"[A-Za-z]", t) for t in texts))]
missing[["v", "type", "data_label", "answer_code", "answer_text"]]


,v,type,data_label,answer_code,answer_text
32,v33,score,Q10 How important for being good citizen: Neve...,"[1, 2, 3, 4, 5, 6, 7]","[1. Not at all important, 2., 3., 4., 5., 6., ..."
33,v34,score,Q11 How important for being good citizen: Alwa...,"[1, 2, 3, 4, 5, 6, 7]","[1. Not at all important, 2., 3., 4., 5., 6., ..."
34,v35,score,Q12 How important for being good citizen: To k...,"[1, 2, 3, 4, 5, 6, 7]","[1. Not at all important, 2., 3., 4., 5., 6., ..."
35,v36,score,Q13 How important for being good citizen: To h...,"[1, 2, 3, 4, 5, 6, 7]","[1. Not at all important, 2., 3., 4., 5., 6., ..."
36,v37,score,Q14 How important for being good citizen: To b...,"[1, 2, 3, 4, 5, 6, 7]","[1. Not at all important, 2., 3., 4., 5., 6., ..."
37,v38,score,Q15 Poorly/well: How well does democracy work ...,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[0. Very poorly, 1., 2., 3., 4., 5., 6., 7., 8..."
38,v41,score,Q18 How important right for democracy: That al...,"[1, 2, 3, 4, 5, 6, 7]","[1. Not at all important, 2., 3., 4., 5., 6., ..."
39,v42,score,Q19 How important right for democracy: That go...,"[1, 2, 3, 4, 5, 6, 7]","[1. Not at all important, 2., 3., 4., 5., 6., ..."
40,v43,score,Q20 How important right for democracy: That pe...,"[1, 2, 3, 4, 5, 6, 7]","[1. Not at all important, 2., 3., 4., 5., 6., ..."
41,v44,score,Q21 How important right for democracy: That ci...,"[1, 2, 3, 4, 5, 6, 7]","[1. Not at all important, 2., 3., 4., 5., 6., ..."


In [75]:
bank[bank['v']=='v33']

,v,source_q,data_label,type,question,codes,options,reversed,note,country_options,country_dropped,answer_code,answer,answer_text
32,v33,Q10,Q10 How important for being good citizen: Neve...,score,There are different opinions as to what it tak...,1 | 2 | 3 | 4 | 5 | 6 | 7,1 Not at all important | 2 | 3 | 4 | 5 | 6 | 7...,False,NaN,NaN,NaN,"[1, 2, 3, 4, 5, 6, 7]","[1. Not at all important, 2, 3, 4, 5, 6, 7. Ve...","[1. Not at all important, 2., 3., 4., 5., 6., ..."


In [76]:
long[long['v']=='v60'].answer.value_counts()

answer
1. Yes    1501
2. No       75
Name: count, dtype: int64

# Export verified

In [77]:
# ---- export for training: one row per answer, with the person's persona, the question and every option offered
long_fn = data_dir/f"issp2023_{country}_long.parquet"
columns = ["CASEID"] + persona + ["v", "label", "question", "type", 
                                  "reversed", "option_codes", "option_texts",
                                  "answer_code", "answer_text"
                                ]
long[columns].to_parquet(long_fn, index=False)
long_fn


PosixPath('/home/dw/github/decision-models-finetuning/data/issp2023/issp2023_US_long.parquet')

In [78]:
bank['options_old'] = bank['options']
bank[['v', 'data_label', 'question', 'options_old', 'answer_code', 'answer_text', 'type', 'reversed']].to_csv(verified_fn,index=False)
